# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 1 — Load, Index and Decompose the Tayyar Series

**Duration** 50 minutes · **Work in pairs** · `git checkout lab1-start`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 5 | Load the raw CSV. Confirm the duplicate and out-of-order timestamps. **Note the row count before cleaning.** |
| 10 | Implement / call `load_demand`: de-duplicate, sort, localise to `Asia/Riyadh`, `asfreq("h")`. Report gap rows and the longest gap. |
| 10 | Impute gaps ≤ 3 h by time interpolation; flag longer gaps. Plot one summer week and one winter week. |
| 10 | Decide additive vs multiplicative with `additive_or_multiplicative`. Justify from the amplitude evidence. |
| 10 | Run MSTL for daily + weekly seasonality. Compute `seasonal_strength` for each. Four-panel figure. |
| 5 | Write three bullets in `FINDINGS.md`. Commit. |

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

### Task 1 — how messy is the raw file?

In [ ]:
raw = pd.read_csv(f"{DATA}/ksa_grid_demand.csv", parse_dates=["timestamp"])
print("raw rows          :", f"{len(raw):,}")
print("duplicate stamps  :", int(raw.timestamp.duplicated().sum()))
print("monotonic         :", raw.timestamp.is_monotonic_increasing)
print("zero demand rows  :", int((raw.demand_mw == 0).sum()))
raw.head()

> **Stop and predict.** How many rows do you expect after cleaning — more, fewer, or the same? Write your answer down before running the next cell.

### Task 2 — the clean, explicitly-indexed series

In [ ]:
from tayyar.data.load import load_demand, load_calendar, join_calendar

# TODO: load the messy CSV into a clean, tz-aware hourly frame.
# Do it in this order — any other order loses data silently:
#   1. parse timestamps      2. drop duplicate timestamps (keep last)
#   3. sort                  4. set_index + tz_localize("Asia/Riyadh")
#   5. asfreq("h")   <-- the line that materialises the gaps
#   6. treat demand_mw == 0 as missing (sensor dropouts)
#
# df = ...

# Then report:
#   rows before / rows after / number of missing hours / longest gap

### Task 3 — a summer week and a winter week

In [ ]:
y = df["demand_mw"].astype(float)

fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharey=True)
y["2023-07-09":"2023-07-16"].plot(ax=ax[0], color="#C00000")
ax[0].set_title("A summer week"); ax[0].set_ylabel("MW")
y["2023-01-08":"2023-01-15"].plot(ax=ax[1], color="#1F3864")
ax[1].set_title("A winter week"); ax[1].set_ylabel("MW")
plt.tight_layout()

# One sentence each, in FINDINGS.md: what is the same, and what is different?

### Task 4 — additive or multiplicative? Decide on evidence.

In [ ]:
from tayyar.analysis.transform import additive_or_multiplicative, apply_transform

# TODO: run additive_or_multiplicative on y and read the level/swing correlation.
# If it says multiplicative, apply the log transform.
# Careful: the series contains sensor zeros. What happens to log(0)?

### Task 5 — MSTL and the seasonal strengths

In [ ]:
from tayyar.analysis.decompose import mstl_decompose

# TODO: MSTL with periods (24, 24*7). Report both seasonal strengths and the trend strength.
# Then build the panel figure.
# HINT: MSTL drops the DatetimeIndex from its components — reattach it before plotting,
#       or every panel will come out blank.

### Task 6 — `FINDINGS.md`

Three bullets, no more:
1. the dominant seasonality and its strength,
2. the trend direction and rough magnitude,
3. what the remainder is telling you (look at Ramadan and at a heatwave week).

```
git add -A && git commit -m "feat(analysis): clean index + MSTL decomposition of Tayyar demand"
```